# TP2 – Exercice 2 : Classification d'images avec Fashion-MNIST
**Module :** Technologies IA – ENSA Fès – Prof. Youness IDRISSI KHAMLICHI

Objectif : construire un ANN puis un CNN, analyser les résultats, comparer les fonctions d'activation.

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Reproductibilité
tf.random.set_seed(42)

(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

class_names = ['T-shirt', 'Pantalon', 'Pull', 'Robe', 'Manteau',
               'Sandale', 'Chemise', 'Basket', 'Sac', 'Bottine']

## Étape 1 – Exploration du jeu de données

In [ ]:
print("Dimensions du train set :", X_train.shape)
print("Dimensions du test set :", X_test.shape)
print("Type :", X_train.dtype)

plt.figure(figsize=(8, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"Label : {y_train[i]} ({class_names[y_train[i]]})")
    plt.axis('off')
plt.tight_layout()
plt.show()

**Question :** les données sont des images en niveaux de gris de 28×28 pixels (60 000 en train, 10 000 en test), 10 classes de vêtements. Les images sont petites et peu détaillées, ce qui rend certaines classes proches (T-shirt / chemise / pull).

## Étape 2 – Prétraitement

In [ ]:
from tensorflow.keras.utils import to_categorical

# Normalisation des pixels entre 0 et 1
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Encodage one-hot des labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

**Réponses**
1. **Normalisation** : met toutes les entrées sur la même échelle, stabilise l'apprentissage et accélère la convergence.
2. **One-hot encoding** : représente chaque classe comme un vecteur (ex. classe 3 → `[0,0,0,1,0,...]`), ce qui évite d'introduire un faux ordre entre classes et est compatible avec `categorical_crossentropy` et la sortie softmax.

## Étape 3 – Modèle dense (ANN)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout

model = Sequential([
    Input(shape=(28, 28)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.2
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='Train acc')
plt.plot(history.history['val_accuracy'], label='Val acc')
plt.legend()
plt.title('Évolution de la précision (ANN)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.legend()
plt.title('Évolution de la perte (ANN)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat)
print("Précision sur le test set :", test_acc)

**Réponses**
1. **ReLU** : simple, rapide à calculer, évite en grande partie le problème de disparition du gradient et donne de bons résultats dans les couches cachées.
2. **accuracy vs val_accuracy** : `accuracy` est mesurée sur les données d'entraînement, `val_accuracy` sur les données de validation (non utilisées pour mettre à jour les poids) ; elle indique la capacité de généralisation.
3. **50 époques** : risque de surapprentissage (accuracy d'entraînement qui monte, validation qui stagne ou baisse), temps de calcul plus long.

## Étape 4 – Comparaison des fonctions d'activation

In [ ]:
from tensorflow.keras.layers import LeakyReLU

activations = ['sigmoid', 'tanh', 'relu']
results = {}

for act in activations:
    model_act = Sequential([
        Input(shape=(28, 28)),
        Flatten(),
        Dense(128, activation=act),
        Dense(64, activation=act),
        Dense(10, activation='softmax')
    ])
    model_act.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    h = model_act.fit(X_train, y_train_cat, epochs=5, batch_size=128,
                      validation_split=0.2, verbose=0)
    results[act] = h.history['val_accuracy'][-1]

# Leaky ReLU
model_leaky = Sequential([
    Input(shape=(28, 28)),
    Flatten(),
    Dense(128),
    LeakyReLU(0.1),
    Dense(64),
    LeakyReLU(0.1),
    Dense(10, activation='softmax')
])
model_leaky.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
h = model_leaky.fit(X_train, y_train_cat, epochs=5, batch_size=128,
                    validation_split=0.2, verbose=0)
results['LeakyReLU'] = h.history['val_accuracy'][-1]

plt.figure(figsize=(7, 4))
plt.bar(results.keys(), results.values())
plt.title("Comparaison des fonctions d'activation")
plt.ylabel("Validation Accuracy")
plt.show()
print(results)

**Réponses**
1. En général ReLU et LeakyReLU donnent les meilleurs résultats (voir le graphique obtenu).
2. **Sigmoïde / tanh** : elles saturent pour de grandes valeurs d'entrée, ce qui provoque la disparition du gradient et un apprentissage lent ; elles sont aussi plus coûteuses à calculer.
3. **LeakyReLU** : laisse passer une petite pente pour les valeurs négatives, ce qui évite les neurones « morts » de ReLU.

## Étape 5 – Modèle CNN

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D

# Ajout de la dimension "canal"
X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_test_cnn = X_test.reshape(-1, 28, 28, 1)

cnn = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])
cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn.summary()

In [ ]:
history_cnn = cnn.fit(
    X_train_cnn, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.2
)

In [ ]:
test_loss, test_acc = cnn.evaluate(X_test_cnn, y_test_cat)
print("Précision CNN sur test set :", test_acc)

plt.figure(figsize=(8, 4))
plt.plot(history_cnn.history['accuracy'], label='Train acc')
plt.plot(history_cnn.history['val_accuracy'], label='Val acc')
plt.legend()
plt.title('Évolution de la précision (CNN)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

**Réponses**
1. **ANN vs CNN** : l'ANN aplatit l'image et traite chaque pixel indépendamment ; le CNN utilise des filtres de convolution qui partagent leurs poids et détectent des motifs locaux (bords, textures, formes).
2. Le CNN conserve la structure spatiale 2D, a moins de paramètres pour une capacité équivalente et est plus robuste aux translations, d'où de meilleurs résultats.
3. **Pooling** : réduit la taille spatiale des cartes de caractéristiques, diminue le nombre de paramètres et le calcul, et apporte une certaine invariance aux petites translations.

## Étape 6 – Visualisation des performances

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = np.argmax(cnn.predict(X_test_cnn), axis=1)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues",
                                        display_labels=class_names, xticks_rotation=45)
plt.title("Matrice de confusion - CNN")
plt.show()

**Réponses**
1. Les classes les plus confondues sont généralement **T-shirt / Chemise / Pull / Manteau** (formes proches), et parfois Basket / Bottine / Sandale.
2. **Réduire les erreurs** : architecture plus profonde, Batch Normalization, data augmentation, plus d'époques avec early stopping, réglage des hyperparamètres.

## Étape 7 – Data Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)
datagen.fit(X_train_cnn)

history_aug = cnn.fit(
    datagen.flow(X_train_cnn, y_train_cat, batch_size=128),
    epochs=10,
    validation_data=(X_test_cnn, y_test_cat)
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_aug.history['accuracy'], label='Train acc')
plt.plot(history_aug.history['val_accuracy'], label='Val acc')
plt.legend()
plt.title('Précision CNN avec Data Augmentation')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

**Réponses**
1. **Data Augmentation** : crée des variantes des images d'entraînement (rotation, zoom, symétrie), ce qui augmente la diversité des données et réduit le surapprentissage.
2. Pour Fashion-MNIST, le **flip horizontal** et les petites rotations / zooms sont adaptés (un vêtement reste reconnaissable). Les rotations fortes ou le flip vertical sont à éviter.

## Analyse personnelle (à compléter après exécution)
- **Interprétation des résultats** : …
- **Difficultés rencontrées** : …
- **Propositions d'amélioration** : plus de couches, BatchNormalization, autres architectures (ResNet), réglage des hyperparamètres.